# 5. Basic Structure Operations

In [3]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [4]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 14:19:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


In [7]:
df = spark.read.format("json").load("../data/flight-data/json/2015-summary.json")
df.printSchema()
df.schema

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



StructType([StructField('DEST_COUNTRY_NAME', StringType(), True), StructField('ORIGIN_COUNTRY_NAME', StringType(), True), StructField('count', LongType(), True)])

## To retain

**Schema on read - We let the data source define the schema**

**The manipulation of columns are called expressions**

**Row objects in Pyspark DF represent internally an array of bytes**

* Columns are just expressions;
* Columns and transformations of those columns compile to the same logical plan as parsed expressions;


Example:
(((col("someCol") + 5) * 200) - 6) < col("otherCol")

![alt text](image.png)

This graph is familiar with the acyclic graph

In [5]:
# How to enforce a schema on a DataFrame

from pyspark.sql.types import StructField, StructType, StringType, LongType
import pyspark.sql.functions as F

myManualSchema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", LongType(), False, metadata={"hello": "world"})
])

df = spark.read.format("json").schema(myManualSchema).load("../data/flight-data/json/2015-summary.json")

F.col("someColumnName")
F.column("someColumnName")


print(df.columns)
df.count()

print(df.first())


# we can create a row
from pyspark.sql import Row
myRow = Row("Hello", None, 1, False)
myRow[0]

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15)


'Hello'

In [6]:
df = spark.read.format("json").load("../data/flight-data/json/2015-summary.json")
df.createOrReplaceTempView("dfTable")


from pyspark.sql import Row
import pyspark.sql.types as T

myManualSchema = T.StructType([
    T.StructField("some", T.StringType(), True),
    T.StructField("col", T.StringType(), True),
    T.StructField("names", T.LongType(), False)
])

myRow = Row("Hello", None, 1)
myDf = spark.createDataFrame([myRow], myManualSchema)
myDf.show()

+-----+----+-----+
| some| col|names|
+-----+----+-----+
|Hello|NULL|    1|
+-----+----+-----+



In [7]:
# Select ant SelectExpr in Dataframes
df.select("DEST_COUNTRY_NAME").show(2)
df.select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").show(2)


df.select(
    F.expr("DEST_COUNTRY_NAME"),
    F.col("DEST_COUNTRY_NAME"),
    F.column("DEST_COUNTRY_NAME")
).show(2)


df.select(F.expr("DEST_COUNTRY_NAME AS destination")).show(2)

# We can manipulate the result of an expression as another expression
df.select(F.expr("DEST_COUNTRY_NAME AS destination")).alias("DEST_COUNTRY_NAME").show(2)

# Since EXPR is so used, Spark created the Shortcut, selectExpr

df.selectExpr("DEST_COUNTRY_NAME as newColumnName", "DEST_COUNTRY_NAME").show(2)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|    United States|
|    United States|
+-----------------+
only showing top 2 rows
+-----------------+-------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|
+-----------------+-------------------+
|    United States|            Romania|
|    United States|            Croatia|
+-----------------+-------------------+
only showing top 2 rows
+-----------------+-----------------+-----------------+
|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|
+-----------------+-----------------+-----------------+
|    United States|    United States|    United States|
|    United States|    United States|    United States|
+-----------------+-----------------+-----------------+
only showing top 2 rows
+-------------+
|  destination|
+-------------+
|United States|
|United States|
+-------------+
only showing top 2 rows
+-------------+
|  destination|
+-------------+
|United States|
|United States|
+-------------+
only showing top

In [8]:
# More advanced logic, how to create a new column with selectExpr
df.selectExpr(
    "*", # get all origin columns
    "(DEST_COUNTRY_NAME = ORIGIN_COUNTRY_NAME) as withinCountry"
).show(2)

# We can also use aggregations
df.selectExpr(
    "avg(count)", "count(distinct(DEST_COUNTRY_NAME))"
).show(2)


# Literals allows us to pass a value from a programming language into a spark value that he can understand
df.select(F.expr("*"), F.lit(1).alias("One")).show(2)

+-----------------+-------------------+-----+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|withinCountry|
+-----------------+-------------------+-----+-------------+
|    United States|            Romania|   15|        false|
|    United States|            Croatia|    1|        false|
+-----------------+-------------------+-----+-------------+
only showing top 2 rows
+-----------+---------------------------------+
| avg(count)|count(DISTINCT DEST_COUNTRY_NAME)|
+-----------+---------------------------------+
|1770.765625|                              132|
+-----------+---------------------------------+

+-----------------+-------------------+-----+---+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|One|
+-----------------+-------------------+-----+---+
|    United States|            Romania|   15|  1|
|    United States|            Croatia|    1|  1|
+-----------------+-------------------+-----+---+
only showing top 2 rows


In [9]:
# we can make spark to be Case Sensitive
spark.conf.set("spark.sql.caseSensitive", "false")

# Adding columns
df.withColumn("numberOne", F.lit(1)).show(2)

df.withColumn("withinCountry", F.expr("ORIGIN_COUNTRY_NAME == DEST_COUNTRY_NAME")).show(2)

# how to rename columns
df.withColumnRenamed("DEST_COUNTRY_NAME", "dest").columns


# We can create columns with Spaces on it
# On this case, there is not need to use ` because the string parameter accepts on the WithColumn
df_with_big_column = df.withColumn("This Long Column-name", F.col("ORIGIN_COUNTRY_NAME"))

df_with_big_column.selectExpr(
    "`This long Column-name`",
    "`This long Column-name` as `new col`"
 ).show(2)


# Cast type columns
df.withColumn("count2", F.col("count").cast("long"))

+-----------------+-------------------+-----+---------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|numberOne|
+-----------------+-------------------+-----+---------+
|    United States|            Romania|   15|        1|
|    United States|            Croatia|    1|        1|
+-----------------+-------------------+-----+---------+
only showing top 2 rows
+-----------------+-------------------+-----+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|withinCountry|
+-----------------+-------------------+-----+-------------+
|    United States|            Romania|   15|        false|
|    United States|            Croatia|    1|        false|
+-----------------+-------------------+-----+-------------+
only showing top 2 rows
+---------------------+-------+
|This long Column-name|new col|
+---------------------+-------+
|              Romania|Romania|
|              Croatia|Croatia|
+---------------------+-------+
only showing top 2 rows


DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint, count2: bigint]

In [10]:
# Filtering rows
# To filter rows, we create an expression that evaluates to True or False

df.filter(F.col("count") < 2).show(2)
df.where("count < 2").show(2)


# Getting unique rows
df.select("ORIGIN_COUNTRY_NAME").distinct().count()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
+-----------------+-------------------+-----+
only showing top 2 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
+-----------------+-------------------+-----+
only showing top 2 rows


125

In [12]:
# Random samples of a dataframe
seed = 5
withReplacement = False
fraction = 0.5
df.sample(withReplacement, fraction, seed).count()

138

In [17]:
# Sometimes in Machine learning algorithms, we need to split our data into training and test datasets. Spark has a method called randomSplit that allows us to do that.

dataFrames = df.randomSplit([0.25, 0.75], seed)
print(len(dataFrames))

dataFrames[0].count() > dataFrames[1].count()
print(dataFrames[0].count())
print(dataFrames[1].count())

2
71
185


## Concatenating and Appending rows
DataFrames are immutable. This means users cannot append to DataFrames because that would be changing it. To append to a DataFrame, you must union the original DataFrame along with the new DataFrame.

**WARNING:** Unions are currently performed based on location, not on schema. This means that columns will not automatically line up the way you think they might.
This means that we should use a select to make sure that all columns align before doing the union

In [18]:
from pyspark.sql import Row

schema = df.schema
newRows = [
    Row("New Country", "Other Country", 5),
    Row("New Country 2", "Other Country 3", 1)
]

parallelizedRows = spark.sparkContext.parallelize(newRows) # we don't need to parallelize to create a Dataframe
newDf = spark.createDataFrame(parallelizedRows, schema)

(
    df
    .union(newDf)
    .where("count = 1")
    .where(F.col("ORIGIN_COUNTRY_NAME") != "United States")
    .show()
)



+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
|    United States|          Gibraltar|    1|
|    United States|             Cyprus|    1|
|    United States|            Estonia|    1|
|    United States|          Lithuania|    1|
|    United States|           Bulgaria|    1|
|    United States|            Georgia|    1|
|    United States|            Bahrain|    1|
|    United States|   Papua New Guinea|    1|
|    United States|         Montenegro|    1|
|    United States|            Namibia|    1|
|    New Country 2|    Other Country 3|    1|
+-----------------+-------------------+-----+



In [ ]:
# By default, the sort is in ascending order

df.sort("count").show(5)
df.orderBy("count", "DEST_COUNTRY_NAME").show(5)
df.orderBy(F.col("count"), F.col("DEST_COUNTRY_NAME")).show(5)

# We can also specify the order of the sorting
df.orderBy(F.expr("count desc")).show(2)
df.orderBy(F.col("count").desc(), F.col("DEST_COUNTRY_NAME").asc()).show(2)

# What happens when we sort one column by DESC and a second column by asc?
# ANSWER: Spark does the first Sort and when it finds columns with the same value, sorts by the second column

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|               Malta|      United States|    1|
|Saint Vincent and...|      United States|    1|
|       United States|            Croatia|    1|
|       United States|          Gibraltar|    1|
|       United States|          Singapore|    1|
+--------------------+-------------------+-----+
only showing top 5 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|     Burkina Faso|      United States|    1|
|    Cote d'Ivoire|      United States|    1|
|           Cyprus|      United States|    1|
|         Djibouti|      United States|    1|
|        Indonesia|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+----

We can use asc_nulls_first, desc_nulls_first, asc_nulls_last or desc_nulls_last to specify where you would like your null values to appear in an ordered DataFrame.

**Very Important**
For optimization purposes, it's sometimes advisable to sort within each partition before another set of transformations. You can use the sortWithinPartitions method to do this.



In [25]:
spark.read.format("json").load("../data/flight-data/json/2015-summary.json").sortWithinPartitions("count")

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint]

## Repartition and Coalesce

Another important optimization opportunity is to partition the data according to some frequently filtered columns, which control the physical layout of the data across the cluster including the partitioning scheme and the number of partitions.

Repartition will incur a fill shuffle of the data, regardless of whether one is necessary. This means that you should typically only repartition when the future number of partitions is greater than your current number of partitions or when you are looking to partition by a set of columns

In [ ]:
# The repartition will incur in a full shuffle
df.rdd.getNumPartitions()
df = df.repartition(5)
df.rdd.getNumPartitions()
# We can also repartition by a column
df = df.repartition(F.col("DEST_COUNTRY_NAME"))
df.rdd.getNumPartitions()

# We can also join these 2 together
df = df.repartition(5, F.col("DEST_COUNTRY_NAME"))
df.rdd.getNumPartitions()

5

In [34]:
# Coalesce will not incur in a full shuffle. This operation will shuffle the data into five partitions based
# on the destination country name, and then coalesce them
df.repartition(5, F.col("DEST_COUNTRY_NAME")).coalesce(2)

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint]

**What is the difference between Coalesce and Repartition?**
Use repartition() when you need better distribution or more partitions. Use coalesce() when only reducing partitions, since the coalesce cannot increase the partitions, often before writing fewer output files.

In [ ]:
# Collecting rows to the Driver
# All of the below methods pull data to the driver
collectDF = df.limit(10)
collectDF.take(5) # take works with an Integer count
collectDF.show() # this prints it out nicely
collectDF.show(5, False)
collectDF.collect()

# There's an additional way of collecting rows to the driver in order to iterate over the entire dataset
# The method toLocalIterator collects partitions to the driver as an iterator.
result = collectDF.toLocalIterator()
# This will print row by row
for item in result:
    print(item)


+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|             Moldova|      United States|    1|
|             Bolivia|      United States|   30|
|             Algeria|      United States|    4|
|Turks and Caicos ...|      United States|  230|
|            Pakistan|      United States|   12|
|    Marshall Islands|      United States|   42|
|            Suriname|      United States|    1|
|              Panama|      United States|  510|
|         New Zealand|      United States|  111|
|             Liberia|      United States|    2|
+--------------------+-------------------+-----+

+------------------------+-------------------+-----+
|DEST_COUNTRY_NAME       |ORIGIN_COUNTRY_NAME|count|
+------------------------+-------------------+-----+
|Moldova                 |United States      |1    |
|Bolivia                 |United States      |30   |
|Algeria                 |United States      |4 

**WARNING**

Any collection of data to the driver can be a very expensive operation! If you have a large dataset and call collect, you can crash the driver. If you use the toLocalIterator and have very large partitions, you can easily crash the driver node and lose the state of your application. This is also expensive because we can operate on a one-by-one basis, instead of running computation in parallel.